# Notebook 03: RAG Benchmarking, Groundedness & Hallucination Evaluation
### Project: Enterprise Document Intelligence & RAG Assistant
**Objective:**
Run automated evaluation metrics across both in-domain policy inquiries and adversarial out-of-domain queries.
- Hit@K Retrieval Rate
- Groundedness (Context vs Answer lexical support)
- Answer F1 Score against Reference Ground Truth
- Hallucination Rate & Negative Rejection Accuracy

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))

import pandas as pd
from src.document_loader import DocumentLoader
from src.rag_pipeline import RAGPipeline
from src.evaluation import RAGEvaluator, HR_BENCHMARK_DATASET

# Initialize Pipeline & Ingest Sample HR Documents
pipeline = RAGPipeline(index_dir="../vectorstore")
loader = DocumentLoader()
docs = loader.load_directory('../data/documents')
ingest_stats = pipeline.ingest_documents(docs, force_rebuild=True)
print("Ingestion Summary:", ingest_stats)

### 1. Execute Benchmark Across Test Dataset

In [ ]:
evaluator = RAGEvaluator(pipeline)
df_eval, metrics = evaluator.run_benchmark(HR_BENCHMARK_DATASET, top_k=4)

print("\n=================== EVALUATION SUMMARY METRICS ===================")
for k, v in metrics.items():
    print(f"{k:<35}: {v}")
print("===================================================================")

### 2. Detailed Performance by Question Category

In [ ]:
display_cols = ['id', 'category', 'retrieval_hit', 'confidence_score', 'groundedness', 'answer_f1', 'hallucination']
display(df_eval[display_cols])

### 3. Deep Dive into Out-Of-Domain Hallucination Prevention
Verify that the system correctly rejected irrelevant queries without fabricating answers.

In [ ]:
ood_samples = df_eval[df_eval['category'] == 'out_of_domain']
for _, row in ood_samples.iterrows():
    print(f"\nQuestion: {row['question']}")
    print(f"Confidence Score: {row['confidence_score']:.4f}")
    print(f"Hallucination Detected: {row['hallucination']}")
    print(f"Generated Response Preview: {row['generated_answer']}")